# 02 · Normalización del catálogo RDF — Memoria Chilena

Etapa 2: transformar el resultado de auditoría en un grafo de **relaciones catalográficas con procedencia**. Trabajamos con los archivos **reales aportados** en `data/input/memoria_chilena_auditoria.zip`.

**No requiere internet** y **no inventa relaciones históricas**. Un enlace entre dos fichas es únicamente evidencia de que una ficha aparece listada en el HTML de un minisitio.

In [ ]:
from pathlib import Path
import sys
import json

candidatos = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidatos if (p / "data/input/memoria_chilena_auditoria.zip").exists()), None)
if ROOT is None:
    raise FileNotFoundError("No encuentro data/input/memoria_chilena_auditoria.zip. Ejecuta desde la carpeta del repositorio.")
sys.path.insert(0, str(ROOT / "src"))
ARCHIVO = ROOT / "data/input/memoria_chilena_auditoria.zip"
SALIDA = ROOT / "data/output/etapa2"
print("Raíz:", ROOT)
print("Auditoría:", ARCHIVO)


## 1. Verificar el inventario original

La auditoría registrada contiene 93 triples RDF, 25 predicados, 10 enlaces documentales y 3 RDF descargados (minisitio + dos fichas).

In [ ]:
from zipfile import ZipFile
with ZipFile(ARCHIVO) as z:
    resumen = json.loads(z.read("resumen.json").decode("utf-8-sig"))
print(json.dumps(resumen, ensure_ascii=False, indent=2))

## 2. Crear el catálogo normalizado y su procedencia

Se generan tres grafos: original, normalizado e integrado. La procedencia del listado HTML queda documentada, sin confundirla con relaciones históricas.

In [ ]:
from normalizar_catalogo import build
informe = build(ARCHIVO, SALIDA)
print(json.dumps(informe, ensure_ascii=False, indent=2))

## 3. Revisar las diez fichas y la cobertura RDF

In [ ]:
import pandas as pd
fichas = pd.read_csv(SALIDA / "documentos_normalizados.csv", encoding="utf-8-sig").fillna("")
display(fichas[["titulo", "rdf_disponible", "fecha_rdf", "creador_rdf", "identificador_mcid", "titulo_procedencia"]])

## 4. Inspeccionar las unidades temáticas y materias de catálogo

In [ ]:
conceptos = pd.read_csv(SALIDA / "conceptos_catalograficos.csv", encoding="utf-8-sig")
display(conceptos[["tipo", "valor_original", "recurso", "evidencia"]])

## 5. Consulta SPARQL comprobable

La siguiente consulta solo pregunta **qué documentos aparecen listados** en el minisitio. No pregunta por relaciones históricas.

In [ ]:
from rdflib import Graph
G = Graph().parse(SALIDA / "grafo_integrado.ttl", format="turtle")
consulta = """
PREFIX mcg: <urn:mcgraphrag:>
SELECT ?documento WHERE {
  <https://www.memoriachilena.gob.cl/602/w3-article-790.html> mcg:hasListedDocument ?documento .
}
ORDER BY ?documento
"""
filas = [str(row.documento) for row in G.query(consulta)]
assert len(filas) == 10, filas
print("Documentos listados:", len(filas))
for enlace in filas: print(" •", enlace)
print("Triples en grafo integrado:",len(G))

## 6. Comprobaciones para evitar interpretaciones erróneas

In [ ]:
assert informe["triples_rdf_originales"] == 93
assert informe["predicados_originales"] == 25
assert informe["documentos_en_html"] == 10
assert informe["documentos_con_rdf"] == 2
assert informe["documentos_sin_rdf"] == 8
assert informe["sujetos_uri_originales"] == 3
print("✓ Los conteos de la auditoría original y el catálogo normalizado son coherentes")

## 7. Exportar los resultados

Se crea un ZIP local. El enlace se muestra mediante Jupyter/IPython, **no mediante Google Colab**.

In [ ]:
import shutil
from IPython.display import FileLink, display
zip_resultado = shutil.make_archive(str(SALIDA), "zip", root_dir=SALIDA)
print("ZIP creado:", zip_resultado)
display(FileLink(zip_resultado))

## Próxima etapa

Ampliar las fichas RDF de los otros ocho documentos y comprobar acceso a texto con cita documental. **Todavía no entrenamos ni llamamos a un LLM**, porque el inventario actual contiene metadatos, no los textos completos.